# DINOv3 + Regression Classifier for TWSI Segmentation
---

This notebook trains a foreground segmentor for tactile walking surface indicators (TWSIs)
by training a patchwise logistic regression classifier on frozen DINOv3 features.

**Pipeline:** Extract DINOv3 patch features → Train LogReg classifier → Inference as foreground segmentation

Based on the [DINOv3 foreground segmentation notebook](https://github.com/facebookresearch/dinov3/blob/main/notebooks/foreground_segmentation.ipynb) by Meta.

**Paper Reference:** GuideTWSI (ICRA 2026), Section VI-A

## 1. Setup

Install dependencies and clone the DINOv3 repository.

In [ ]:
!git clone https://github.com/facebookresearch/dinov3.git

## 2. Configuration

Load training hyperparameters from the config file and set paths.

**Important:** To use DINOv3, sign up and accept the terms of use from Meta to obtain
checkpoint download links.

In [ ]:
import yaml

with open("../configs/dinov3_regcls.yaml", "r") as f:
    config = yaml.safe_load(f)

print("Config loaded:")
for section, values in config.items():
    print(f"  {section}: {values}")

In [ ]:
# Set paths
REPO_DIR = "dinov3"  # Path where DINOv3 repo was cloned
DINOv3_MODEL_URL = ""  # Add your DINOv3 checkpoint download link here

# Model config from YAML
MODEL_NAME = config["model"]["dinov3"]["variant"]  # dinov3_vits16
PATCH_SIZE = config["training"]["patch_size"]  # 16
IMAGE_SIZE = config["training"]["image_size"]  # 768
CLASSIFIER_C = config["training"]["classifier_C"]  # 0.1
MAX_ITER = config["training"]["max_iterations"]  # 100000
NEG_THRESH = config["training"]["negative_threshold"]  # 0.01
POS_THRESH = config["training"]["positive_threshold"]  # 0.99
IMAGENET_MEAN = tuple(config["training"]["normalize_mean"])
IMAGENET_STD = tuple(config["training"]["normalize_std"])
N_LAYERS = config["model"]["dinov3"]["num_layers"]  # 12
CONFIDENCE_THRESHOLD = config["inference"]["confidence_threshold"]  # 0.5

## 3. Download Dataset

Download the GuideTWSI dataset from HuggingFace Hub.

In [ ]:
# Download dataset from HuggingFace
# !huggingface-cli download guidedogrobot-tactile/GuideTWSI --local-dir ./data

# Set data paths
# Expected structure: data/images/*.jpg + data/masks/*.png
IMAGES_URI = "data/train/images"  # Path to training images
LABELS_URI = "data/train/masks"   # Path to training masks
TEST_IMAGES_URI = "data/test/images"
TEST_MASKS_URI = "data/test/masks"

## 4. Load Data

In [ ]:
import io
import os
import pickle

from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from sklearn.linear_model import LogisticRegression
import torch
import torchvision.transforms.functional as TF
from tqdm import tqdm


def load_images(uri: str) -> list:
    images = []
    for filename in sorted(os.listdir(uri)):
        if filename.endswith(".jpg") or filename.endswith(".png"):
            image = Image.open(os.path.join(uri, filename))
            images.append(image)
    return images


images = load_images(IMAGES_URI)
labels = load_images(LABELS_URI)
n_images = len(images)
assert n_images == len(labels), f"{len(images)=}, {len(labels)=}"

print(f"Loaded {n_images} images and labels")

## 5. Load DINOv3 Model

In [ ]:
model = torch.hub.load(REPO_DIR, MODEL_NAME, source="local", weights=DINOv3_MODEL_URL)
model.cuda()
print(f"Loaded {MODEL_NAME} with {N_LAYERS} layers, patch size {PATCH_SIZE}")

## 6. Extract Features

Build a per-patch label map by quantizing ground truth masks to the 16×16 patch grid,
then extract DINOv3 features for all training images.

**Note:** This is memory-intensive. Use a GPU with sufficient VRAM or reduce batch size.

In [ ]:
# Quantization filter for the given patch size
patch_quant_filter = torch.nn.Conv2d(1, 1, PATCH_SIZE, stride=PATCH_SIZE, bias=False)
patch_quant_filter.weight.data.fill_(1.0 / (PATCH_SIZE * PATCH_SIZE))


def resize_transform(mask_image, image_size=IMAGE_SIZE, patch_size=PATCH_SIZE):
    w, h = mask_image.size
    h_patches = int(image_size / patch_size)
    w_patches = int((w * image_size) / (h * patch_size))
    return TF.to_tensor(TF.resize(mask_image, (h_patches * patch_size, w_patches * patch_size)))


xs = []
ys = []
image_index = []

with torch.inference_mode():
    with torch.autocast(device_type="cuda", dtype=torch.float32):
        for i in tqdm(range(n_images), desc="Extracting features"):
            # Load and quantize ground truth
            mask_i = labels[i].split()[-1]
            mask_i_resized = resize_transform(mask_i)
            mask_i_quantized = patch_quant_filter(mask_i_resized).squeeze().view(-1).detach().cpu()
            ys.append(mask_i_quantized)

            # Load and normalize image
            image_i = images[i].convert("RGB")
            image_i_resized = resize_transform(image_i)
            image_i_resized = TF.normalize(image_i_resized, mean=IMAGENET_MEAN, std=IMAGENET_STD)
            image_i_resized = image_i_resized.unsqueeze(0).cuda()

            # Extract features from all layers
            feats = model.get_intermediate_layers(
                image_i_resized, n=range(N_LAYERS), reshape=True, norm=True
            )
            dim = feats[-1].shape[1]
            xs.append(feats[-1].squeeze().view(dim, -1).permute(1, 0).detach().cpu())
            image_index.append(i * torch.ones(ys[-1].shape))

# Concatenate into tensors
xs = torch.cat(xs)
ys = torch.cat(ys)
image_index = torch.cat(image_index)

# Keep only patches with clear positive/negative labels
idx = (ys < NEG_THRESH) | (ys > POS_THRESH)
xs = xs[idx]
ys = ys[idx]
image_index = image_index[idx]

print(f"Design matrix: {xs.shape}")
print(f"Label matrix: {ys.shape}")

## 7. Train Classifier

Train a logistic regression classifier on the extracted DINOv3 features.

Hyperparameters from paper: `C=0.1`, `max_iter=100,000`

In [ ]:
clf = LogisticRegression(
    random_state=0, C=CLASSIFIER_C, max_iter=MAX_ITER, verbose=2
).fit(xs.numpy(), (ys > 0).long().numpy())

print("Classifier trained successfully.")

In [ ]:
# Save classifier
save_path = "dinov3_regcls_classifier.pkl"
with open(save_path, "wb") as f:
    pickle.dump(clf, f)
print(f"Classifier saved to {save_path}")

## 8. Inference

Run inference on a test image and visualize the segmentation result.

In [ ]:
# Load classifier (if starting from a saved checkpoint)
# with open("dinov3_regcls_classifier.pkl", "rb") as f:
#     clf = pickle.load(f)

# Select a test image
test_images = sorted(os.listdir(TEST_IMAGES_URI))
test_image_fpath = os.path.join(TEST_IMAGES_URI, test_images[0])
test_mask_fpath = os.path.join(TEST_MASKS_URI, test_images[0].replace(".jpg", ".png"))

test_image = Image.open(test_image_fpath)
test_image_resized = resize_transform(test_image)
test_image_normalized = TF.normalize(test_image_resized, mean=IMAGENET_MEAN, std=IMAGENET_STD)
test_mask = Image.open(test_mask_fpath)

# Extract features
with torch.inference_mode():
    with torch.autocast(device_type="cuda", dtype=torch.float32):
        feats = model.get_intermediate_layers(
            test_image_normalized.unsqueeze(0).cuda(), n=range(N_LAYERS), reshape=True, norm=True
        )
        x = feats[-1].squeeze().detach().cpu()
        dim = x.shape[0]
        x = x.view(dim, -1).permute(1, 0)

h_patches, w_patches = [int(d / PATCH_SIZE) for d in test_image_resized.shape[1:]]

# Classify patches
fg_score = clf.predict_proba(x)[:, 1].reshape(h_patches, w_patches)
binary_mask = (np.asarray(fg_score) > CONFIDENCE_THRESHOLD).astype(np.uint8) * 255

# Visualize
plt.figure(figsize=(12, 4), dpi=150)
plt.subplot(1, 3, 1)
plt.axis("off")
plt.imshow(test_image_resized.permute(1, 2, 0))
plt.title("Input Image")
plt.subplot(1, 3, 2)
plt.axis("off")
plt.imshow(binary_mask, cmap="gray")
plt.title("Prediction")
plt.subplot(1, 3, 3)
plt.axis("off")
plt.imshow(test_mask, cmap="gray")
plt.title("Ground Truth")
plt.tight_layout()
plt.show()

## 9. Evaluation

Evaluate the model on the full test set using the metrics from `evaluation/metrics.py`.

In [ ]:
import sys
sys.path.insert(0, "..")
from evaluation.metrics import compute_binary_metrics, aggregate_metrics

all_metrics = []
test_images_list = sorted(os.listdir(TEST_IMAGES_URI))

for img_name in tqdm(test_images_list, desc="Evaluating"):
    img_path = os.path.join(TEST_IMAGES_URI, img_name)
    mask_path = os.path.join(TEST_MASKS_URI, img_name.replace(".jpg", ".png"))
    if not os.path.exists(mask_path):
        continue

    test_img = Image.open(img_path).convert("RGB")
    gt_mask = np.array(Image.open(mask_path).convert("L"))
    test_resized = resize_transform(test_img)
    test_norm = TF.normalize(test_resized, mean=IMAGENET_MEAN, std=IMAGENET_STD)

    with torch.inference_mode():
        with torch.autocast(device_type="cuda", dtype=torch.float32):
            feats = model.get_intermediate_layers(
                test_norm.unsqueeze(0).cuda(), n=range(N_LAYERS), reshape=True, norm=True
            )
            x = feats[-1].squeeze().detach().cpu()
            dim = x.shape[0]
            x = x.view(dim, -1).permute(1, 0)

    h_p, w_p = [int(d / PATCH_SIZE) for d in test_resized.shape[1:]]
    fg = clf.predict_proba(x)[:, 1].reshape(h_p, w_p)
    pred = (fg > CONFIDENCE_THRESHOLD).astype(np.uint8) * 255
    pred_resized = np.array(Image.fromarray(pred).resize((gt_mask.shape[1], gt_mask.shape[0]), Image.NEAREST))

    metrics = compute_binary_metrics(gt_mask, pred_resized)
    all_metrics.append(metrics)

avg = aggregate_metrics(all_metrics)
print(f"\nResults over {len(all_metrics)} test images:")
for k, v in avg.items():
    print(f"  {k}: {v:.4f}")